## Step 1: Import the dataset

Loaded the raw Netflix dataset with Pandas and inspected its structure.

**Findings:**
- 8,790 rows and 10 columns

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#load the raw netflix dataset
df = pd.read_csv("../data/raw/Dataset.csv")

print("Shape:", df.shape)
df.info()
df.head()

Shape: (8790, 10)
<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 686.8 KB


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


## Step 2: Identify and handle missing values

`.isnull()` reports zero missing values, but missing data was hidden as the text `"Not Given"`.

| Column | Hidden missing values |
|---|---|
| director | 2,588 (~29%) |
| country | 287 (~3%) |

**Decision:** Converted `"Not Given"` to NaN, then labelled it `"Unknown"` instead of dropping rows. Dropping them would remove about 29% of the data, and `director` is not needed for the later analysis. `"Unknown"` will be excluded from country rankings in Task 3.

In [2]:
#.isnull() only catches real NaN cells
print("Real nulls:\n", df.isnull().sum(), "\n")

#hidden missing values stored as text
print("Hidden 'Not Given' values:\n", (df == "Not Given").sum())

Real nulls:
 show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64 

Hidden 'Not Given' values:
 show_id            0
type               0
title              0
director        2588
country          287
date_added         0
release_year       0
rating             0
duration           0
listed_in          0
dtype: int64


## Step 3: Remove duplicate records and formatting inconsistencies

Duplicates were checked on every column **except** `show_id`, since each record has a unique ID even when its content is identical.

**Findings:**
- 3 duplicates were visible straight away (`15-Aug`, `22-Jul`, `9-Feb`)
- A 4th (`Consequences`) was hidden by stray whitespace
- Stripping whitespace **before** de-duplicating found all **4**

**Formatting fixes:**
- Stripped leading and trailing spaces in all text columns
- Converted `date_added` from text to `datetime`
- Created `year_added` for use in trend analysis (Task 4)

**Result:** 8,790 → **8,786 rows**

In [3]:
#convert the "Not Given" placeholder to proper NaN
df = df.replace("Not Given", np.nan)

#label missing values clearly instead of dropping ~29% of the rows
df["director"] = df["director"].fillna("Unknown")
df["country"] = df["country"].fillna("Unknown")

#verify
print("Remaining nulls:\n", df.isnull().sum(), "\n")
print("Unknown directors:", (df["director"] == "Unknown").sum())
print("Unknown countries:", (df["country"] == "Unknown").sum())

Remaining nulls:
 show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64 

Unknown directors: 2588
Unknown countries: 287


In [4]:
#exact duplicates ignoring show_id
cols = [c for c in df.columns if c != "show_id"]
print("Duplicate records:", df.duplicated(subset=cols).sum())

#look at them
df[df.duplicated(subset=cols, keep=False)].sort_values("title")

Duplicate records: 3


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
2925,s3963,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
4261,s5967,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
3285,s4523,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
4260,s5966,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
393,s3997,TV Show,9-Feb,Unknown,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"
537,s5965,TV Show,9-Feb,Unknown,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"


In [5]:
#strip stray whitespace first
for c in df.select_dtypes(include=["object", "str"]).columns:
    df[c] = df[c].str.strip()

#then remove duplicates
cols = [c for c in df.columns if c != "show_id"]
before = len(df)
df = df.drop_duplicates(subset=cols, keep="first").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicates. New shape: {df.shape}")

#convert date_added to a real date
df["date_added"] = pd.to_datetime(df["date_added"])
df["year_added"] = df["date_added"].dt.year

print(df[["date_added", "year_added"]].dtypes)

Removed 4 duplicates. New shape: (8786, 10)
date_added    datetime64[us]
year_added             int32
dtype: object


In [6]:
print("Types:\n", df["type"].value_counts(), "\n")
print("Ratings:\n", df["rating"].value_counts(), "\n")
print("Unique countries:", df["country"].nunique())

Types:
 type
Movie      6123
TV Show    2663
Name: count, dtype: int64 

Ratings:
 rating
TV-MA       3204
TV-14       2155
TV-PG        861
R            798
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64 

Unique countries: 86


## Step 4: Standardize Country, Rating and Type

| Column | Check | Action |
|---|---|---|
| `type` | Only `Movie` and `TV Show` | Converted to a category |
| `country` | 86 unique values, no spelling or case clashes | Applied consistent title case |
| `rating` | `NR` and `UR` both mean "not rated" | Merged into `Not Rated` (82 rows) |

**New column:** `audience` groups ratings into Kids, Older Kids, Teens, Adults and Not Rated. It will be reused for the rating analysis in Task 5.

In [7]:
#country consistent title-case text
df["country"] = df["country"].str.strip().str.title()

#type restrict to two categories
df["type"] = df["type"].str.strip().astype("category")

#rating merge NR and UR into one "Not Rated" category
df["rating"] = df["rating"].replace({"NR": "Not Rated", "UR": "Not Rated"})

#audience group
audience_map = {
    "TV-Y": "Kids", "TV-Y7": "Kids", "TV-Y7-FV": "Kids", "TV-G": "Kids", "G": "Kids",
    "TV-PG": "Older Kids", "PG": "Older Kids",
    "TV-14": "Teens", "PG-13": "Teens",
    "TV-MA": "Adults", "R": "Adults", "NC-17": "Adults",
    "Not Rated": "Not Rated",
}
df["audience"] = df["rating"].map(audience_map)

print(df["rating"].value_counts(), "\n")
print(df["audience"].value_counts(dropna=False))

rating
TV-MA        3204
TV-14        2155
TV-PG         861
R             798
PG-13         490
TV-Y7         333
TV-Y          306
PG            287
TV-G          220
Not Rated      82
G              41
TV-Y7-FV        6
NC-17           3
Name: count, dtype: int64 

audience
Adults        4005
Teens         2645
Older Kids    1148
Kids           906
Not Rated       82
Name: count, dtype: int64


In [8]:
print("Rows:", len(df))
print("Sum of rating counts:", df["rating"].value_counts().sum())
print("Null ratings:", df["rating"].isnull().sum())
print("TV-MA:", (df["rating"] == "TV-MA").sum())

Rows: 8786
Sum of rating counts: 8786
Null ratings: 0
TV-MA: 3204


## Step 5: Export the cleaned dataset

Saved to `data/processed/netflix_cleaned.csv`.

**Final dataset:**
- **8,786 rows × 12 columns** (10 original + `year_added` + `audience`)
- 0 missing values
- 0 duplicates
- Ready for Tasks 2–4

**Summary of changes:** 4 duplicates removed, hidden missing values labelled, dates converted, ratings merged, audience groups added.

In [9]:
df.to_csv("../data/processed/netflix_cleaned.csv", index=False)
#verification that the file is saved
check = pd.read_csv("../data/processed/netflix_cleaned.csv")
print("Saved shape:", check.shape)
print("Nulls:", check.isnull().sum().sum())

Saved shape: (8786, 12)
Nulls: 0
